In [ ]:
# %% [Block 0 — Setup]
# DuckDB is our SQL engine. We query files (Parquet, CSV, Excel) directly,
# without first loading them into pandas. pandas is only used here to PREPARE
# the files (the data-prep step below) and to receive query results as DataFrames.

import time
from pathlib import Path

import numpy as np
import pandas as pd
import duckdb

RAW_DIR = set the directory here      # where the original .dta files live

OUT_DIR = set the directory here      # where we write the converted files
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Output paths for the formats we will query with DuckDB
PERSONS    = OUT_DIR / 'persons.parquet'     # largest file -> Parquet
HOUSEHOLDS = OUT_DIR / 'households.parquet'  # also large   -> Parquet
CSV        = OUT_DIR / 'geography.csv'        # small file   -> CSV

print('DuckDB version:', duckdb.__version__)

In [ ]:
# %% [Block 1 — DATA PREP (not part of the exercise)]
# One-off step: read the three Stata files the SAME way as the visualization
# exercise (pandas.read_stata, value labels arrive as strings) and save each one
# to a different on-disk format. We deliberately DO NOT cast anything to int here
# — any numeric conversion happens later, inside SQL, with TRY_CAST.
#
# Format choice (see 7.3): Parquet for the two large files (Persons and
# Households), CSV for the small Geography file. Households has more rows than
# Excel's ~1,048,576-row limit, so Parquet is the right home for it too.

# --- Small + medium files fit in memory: read whole, write out ---
geo = pd.read_stata(RAW_DIR / 'Census2022Geography.dta')
hh  = pd.read_stata(RAW_DIR / 'Census2022Households.dta')

geo.to_csv(CSV, index=False)            # Geography -> CSV

# Households is large too: convert its labelled 'category' columns to strings
# (keeps the label text, no int casting) and write to Parquet.
hh_cat = hh.select_dtypes('category').columns
hh[hh_cat] = hh[hh_cat].astype('string')
hh.to_parquet(HOUSEHOLDS, index=False)  # Households -> Parquet

# --- Persons is the largest (~300 MB .dta): stream it in chunks straight to
# Parquet so we never hold the whole file in memory. We convert the labelled
# 'category' columns to plain strings first, which keeps the label TEXT (no
# integer casting) and gives every chunk an identical schema to append. ---
import pyarrow as pa
import pyarrow.parquet as pq

t0 = time.time()
writer = None
rows = 0
with pd.read_stata(RAW_DIR / 'Census2022Persons.dta', chunksize=100_000) as reader:
    for chunk in reader:
        # category -> string: preserves the label text, stabilises the schema
        cat_cols = chunk.select_dtypes('category').columns
        chunk[cat_cols] = chunk[cat_cols].astype('string')

        table = pa.Table.from_pandas(chunk, preserve_index=False)
        if writer is None:
            writer = pq.ParquetWriter(PERSONS, table.schema)
        writer.write_table(table)
        rows += len(chunk)

        break

if writer is not None:
    writer.close()

print(f'Persons -> Parquet: {rows:,} rows in {time.time() - t0:.1f}s')
# A ~300 MB .dta typically converts in well under a minute; the resulting
# Parquet is far smaller and much faster for DuckDB to scan.

In [ ]:
# %% [Task 0 — Open the different files in DuckDB]
# One connection, two formats. Nothing is loaded into pandas: DuckDB reads
# each file from disk and hands us back a small DataFrame with .df().

con = duckdb.connect()

# Peek at each file (LIMIT keeps the preview tiny)
print('--- Persons (Parquet) ---')
print(con.sql(f"SELECT * FROM read_parquet('{PERSONS}') LIMIT 5").df())

print('--- Households (Parquet) ---')
#  your code here

print('--- Geography (CSV) ---')
#  your code here

In [ ]:
# %% [Task 1 — A simple SELECT]
# Column names carry underscores and capitals, so we double-quote every
# identifier. This is the SQL equivalent of persons[["PID", "P02_SEX", ...]].

#  your code here

In [ ]:
# %% [Task 2 — Filtering, and a parameterized query]
# WHERE is the SQL row filter (like a pandas boolean mask). P04_AGE is stored as
# label TEXT, so we TRY_CAST it to a number inside SQL before comparing it.

# (a) Working-age women, born outside the Western Cape
#  your code here

# (b) Same idea, but PARAMETERIZED — no user values baked into the SQL string.
# DuckDB fills each '?' from the params list. This is the safe way to inject
# values (a file path, a threshold, a category) into a query.
min_age = 18
group = 'Black African'

#  your code here

In [ ]:
# %% [Task 3 — GROUP BY: repeating chart transformations in SQL]
# These reproduce, in one SQL statement each, aggregations we built with pandas
# in the visualization exercise.

# (a) Households interviewed by province  (cf. "Households by province").
#     Source: the Geography CSV.
#  your code here

# (b) Internet access by household-head population group
#     (cf. "Internet access by population group"). Source: the Households Parquet
#     file. We turn the internet column into a Yes/No flag with CASE.
#  your code here

In [ ]:
# %% [Task 4 — Join across formats: Parquet (persons) + CSV (geography)]
# Persons is many-to-one to Geography on QID. DuckDB joins the Parquet file and
# the CSV file in a single query — one engine, two formats. We CAST both keys to
# VARCHAR so the join key types match regardless of how each format stored QID.

#  your code here

# Join sanity check: the joined person count should not exceed the number of
# persons we started with — if it does, the join key has duplicates.
#  your code here

In [ ]:
# %% [Task 5 — Modify data: add a column, update rows]
# Reading files is read-only. To CHANGE data we load it into a real DuckDB
# table, edit it there, then write a NEW file back out (the original file is
# never edited in place).

# 1) Materialise the households Parquet file as an in-database table
con.execute(f"CREATE OR REPLACE TABLE households AS SELECT * FROM read_parquet('{HOUSEHOLDS}')")

# 2) Add a new column and populate it (a simple derived flag)
#  your code here

# 3) Add an asset-count index (how many of these durables the household owns)
con.execute('ALTER TABLE households ADD COLUMN asset_index INTEGER')
#  your code here

# 4) Update rows: treat the 'Unspecified' tenure sentinel as a proper missing value
con.execute("""UPDATE households SET "H03_TENURE" = NULL WHERE "H03_TENURE" = 'Unspecified'""")

# 5) Inspect the result
#  your code here

In [ ]:
# %% [Task 6 — Save the modified data back to disk]
# Write the enriched table to Parquet with DuckDB's COPY. Files are immutable,
# so this creates a new file rather than editing the source in place.
con.execute(f"COPY households TO '{OUT_DIR / 'households_enriched.parquet'}' (FORMAT parquet)")

# The full table is far larger than Excel's ~1,048,576-row limit, so we keep it
# in Parquet. Stakeholder deliverables are usually AGGREGATES, which are small
# enough for Excel — build the summary first, then export that.
#  your code here

summary.to_excel(OUT_DIR / 'households_summary.xlsx', index=False)

print('Saved households_enriched.parquet and households_summary.xlsx')
# Homework: add a column `head_age_band` that buckets DERH_HHAGE into
# '<30', '30-49', '50-64', '65+' using a CASE expression, then save back.